# Lecture 11: Data Visualisation and Ethics in Data Science
**BITE 485 | Tharaka University**

---

## Learning Outcomes
1. Apply Tufte's principles to create high data-ink visualisations
2. Select the right chart type for the analytical question
3. Detect algorithmic bias by evaluating performance across groups
4. Apply explainability tools (SHAP-style feature importance)
5. Articulate the ethical responsibilities of data scientists

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
print('Ready.')

## 11.1 Chart Selection — The Right Tool for the Question

Different analytical questions demand different chart types.

In [ ]:
np.random.seed(42)
# Simulate COVID-19 case data for 5 African countries over 12 months
months = pd.date_range('2021-01', periods=12, freq='MS')
countries = ['Kenya','Ethiopia','Nigeria','South Africa','Ghana']
data = {c: np.cumsum(np.random.poisson(lam=np.random.randint(200,800), size=12))
        for c in countries}
df_covid = pd.DataFrame(data, index=months)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. Line chart — trends over time
df_covid.plot(ax=axes[0,0], linewidth=2)
axes[0,0].set_title('Cumulative COVID-19 Cases (Line Chart)')
axes[0,0].set_ylabel('Cumulative Cases')
axes[0,0].tick_params(axis='x', rotation=30)

# 2. Bar chart — comparison at a point in time
final = df_covid.iloc[-1]
axes[0,1].bar(final.index, final.values, color='steelblue', edgecolor='white')
axes[0,1].set_title('Total Cases by Country — Dec 2021 (Bar Chart)')
axes[0,1].set_ylabel('Cases')

# 3. Scatter — relationship
df_scatter = pd.DataFrame({
    'gdp_per_capita': np.random.uniform(500, 15000, 50),
})
df_scatter['cases_per_million'] = 1000 + 0.05*df_scatter['gdp_per_capita'] + np.random.normal(0,500,50)
sns.scatterplot(data=df_scatter, x='gdp_per_capita', y='cases_per_million', ax=axes[0,2])
axes[0,2].set_title('GDP vs Cases per Million (Scatter)')

# 4. Histogram — distribution
new_cases = np.random.poisson(lam=400, size=365)
axes[1,0].hist(new_cases, bins=30, color='tomato', edgecolor='white')
axes[1,0].axvline(np.mean(new_cases), color='blue', linestyle='--', label='Mean')
axes[1,0].set_title('Daily New Cases Distribution (Histogram)')
axes[1,0].legend()

# 5. Box plot — group comparison
df_bp = pd.DataFrame({
    'country': np.repeat(countries, 50),
    'daily_cases': np.concatenate([np.random.poisson(lam=np.random.randint(100,600), size=50)
                                   for _ in countries])
})
sns.boxplot(data=df_bp, x='country', y='daily_cases', ax=axes[1,1], palette='Set2')
axes[1,1].set_title('Daily Cases by Country (Box Plot)')
axes[1,1].tick_params(axis='x', rotation=20)

# 6. Heatmap — correlation/matrix
corr = df_covid.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, ax=axes[1,2])
axes[1,2].set_title('Country Case Correlation (Heatmap)')

plt.suptitle('Chart Type Selection: Matching Visualisation to Question', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 11.2 Good vs Bad Visualisation — The Data-Ink Ratio

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
categories = ['Nairobi','Mombasa','Kisumu','Nakuru','Eldoret']
values = [42, 18, 12, 15, 8]

# BAD: low data-ink ratio
axes[0].bar(categories, values, color=['red','blue','green','orange','purple'],
            width=0.9, edgecolor='black', linewidth=2)
for i, v in enumerate(values): axes[0].text(i, v+0.5, f'{v}%', ha='center', fontweight='bold')
axes[0].set_facecolor('#EEEEEE')
axes[0].grid(True, color='white', linewidth=2)
axes[0].set_title('BAD: Low Data-Ink Ratio', fontsize=12, color='red')
axes[0].set_ylabel('Market Share (%)')
for spine in axes[0].spines.values(): spine.set_visible(True)

# GOOD: high data-ink ratio
axes[1].barh(categories, values, color='steelblue', edgecolor='white')
for i, v in enumerate(values): axes[1].text(v+0.3, i, f'{v}%', va='center')
for spine in ['top','right','left']:
    axes[1].spines[spine].set_visible(False)
axes[1].xaxis.set_visible(False)
axes[1].set_title('GOOD: High Data-Ink Ratio', fontsize=12, color='green')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 11.3 Algorithmic Bias Detection

**Scenario:** A credit scoring model performs differently across gender and location.

In [ ]:
np.random.seed(42)
n = 2000
# Simulate biased training data
gender   = np.random.choice(['M', 'F'], n)
location = np.random.choice(['Urban', 'Rural'], n)
income   = np.where(gender=='M', np.random.normal(55000, 20000, n),
                                  np.random.normal(45000, 18000, n))
credit_score = np.random.randint(300, 850, n)
debt_ratio   = np.random.uniform(0.1, 0.8, n)

# Biased label: historical data discriminates against Rural and Female
approved = (
    (income > 50000).astype(int) * 0.5 +
    (credit_score > 650).astype(int) * 0.3 +
    (debt_ratio < 0.4).astype(int) * 0.2 +
    (gender == 'M').astype(int) * 0.15 +     # gender bias
    (location == 'Urban').astype(int) * 0.1  # location bias
)
approved = (approved + np.random.normal(0, 0.1, n) > 0.7).astype(int)

df_credit = pd.DataFrame({
    'gender': gender, 'location': location,
    'income': income, 'credit_score': credit_score,
    'debt_ratio': debt_ratio, 'approved': approved
})

print(f'Overall approval rate: {df_credit.approved.mean()*100:.1f}%')
print('\nApproval rate by gender:')
print(df_credit.groupby('gender').approved.mean().mul(100).round(1).to_string())
print('\nApproval rate by location:')
print(df_credit.groupby('location').approved.mean().mul(100).round(1).to_string())

In [ ]:
# Train a model on biased data
from sklearn.preprocessing import LabelEncoder
df_model = df_credit.copy()
df_model['gender_enc']   = LabelEncoder().fit_transform(df_model['gender'])
df_model['location_enc'] = LabelEncoder().fit_transform(df_model['location'])

features = ['income', 'credit_score', 'debt_ratio', 'gender_enc', 'location_enc']
X = df_model[features]
y = df_model['approved']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)
rf_credit = RandomForestClassifier(100, random_state=42)
rf_credit.fit(X_tr, y_tr)

df_test = df_credit.iloc[X_te.index].copy()
df_test['prediction'] = rf_credit.predict(X_te)

print('Model performance by gender:')
for g in ['M', 'F']:
    sub = df_test[df_test.gender == g]
    acc = (sub.prediction == sub.approved).mean()
    recall_approved = ((sub.prediction==1) & (sub.approved==1)).sum() / (sub.approved==1).sum()
    print(f'  {g}: Accuracy={acc:.3f}, Recall (approved)={recall_approved:.3f}')

print('\nModel performance by location:')
for loc in ['Urban', 'Rural']:
    sub = df_test[df_test.location == loc]
    acc = (sub.prediction == sub.approved).mean()
    recall_approved = ((sub.prediction==1) & (sub.approved==1)).sum() / (sub.approved==1).sum()
    print(f'  {loc}: Accuracy={acc:.3f}, Recall (approved)={recall_approved:.3f}')

## 11.4 Feature Importance as a Transparency Tool

In [ ]:
fi = pd.DataFrame({'feature': features,
                   'importance': rf_credit.feature_importances_})\
       .sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['tomato' if f in ['gender_enc','location_enc'] else 'steelblue' for f in fi.feature]
ax.barh(fi.feature[::-1], fi.importance[::-1], color=colors[::-1])
ax.set_xlabel('Feature Importance')
ax.set_title('Credit Model Feature Importance\n(Red = protected attributes)')
for spine in ['top','right']: ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()
print('\nEthical concern: gender and location are in the top features.')
print('A fair model should not rely on protected characteristics.')

## 11.5 Ethical Reflection

In [ ]:
print('=== ETHICAL CHECKLIST FOR THIS CREDIT MODEL ===')
checks = [
    ('Data bias', 'Training data reflects historical discrimination against women and rural applicants.'),
    ('Protected attributes', 'Gender and location are used as features — direct discrimination.'),
    ('Disparate impact', 'Rural female applicants are approved at a significantly lower rate.'),
    ('Transparency', 'Model is a black-box random forest — difficult to explain individual decisions.'),
    ('Accountability', 'Who is responsible when a qualified applicant is denied due to model bias?'),
    ('Regulatory risk', 'Kenya Data Protection Act and fair lending principles prohibit discriminatory scoring.'),
]
for issue, description in checks:
    print(f'\n[!] {issue.upper()}')
    print(f'    {description}')

### Exercise

Bias mitigation exercise:

1. Retrain the credit model WITHOUT gender_enc and location_enc as features.
2. Compare the approval rates by gender and location in the new model vs the original.
3. Compare the overall accuracy of both models. Is there a significant trade-off?
4. Write a 150-word ethical reflection on the following question: 'A bank argues that including location improves model accuracy. Should accuracy improvements justify using location as a feature in credit scoring? Explain your reasoning.'

In [ ]:
# Your code here


---
*BITE 485 Data Science | Tharaka University | kevin.tuei@tharaka.ac.ke*

## Course Complete!

You have now worked through all 11 lectures of BITE 485 Data Science. The skills covered in this course span the full data science workflow:

- Data collection and exploration (L1, L4)
- Statistical foundation (L2, L3)
- Machine learning (L5, L6, L7)
- Feature engineering and model tuning (L8, L9)
- Specialised topics (L10 Graph Mining, L11 Ethics)

Remember: **good data science is not just technically correct — it is honest, fair, and accountable.**